# 06-05. Ideal Age in 7-box model  
# 7-box model で Ideal Age を計算する

**Ocean Box Modeling with Python / 海洋ボックスモデル入門**

## 今日の目的 / Goals

06-04 では、7-box model の保存トレーサー実験を行いました。  
In 06-04, we performed passive tracer experiments in 7-box model.

この Notebook では、**Ideal Age** を導入します。  
In this notebook, we introduce **Ideal Age**.

Ideal Age は、

```text
最後に表層と接してからの時間
```

を表すトレーサーです。  
Ideal Age is a tracer representing:

```text
time since last contact with the surface
```

7-box model では、特に **A と D の年齢差** が重要です。  
In 7-box model, the age difference between **A and D** is especially important.

> **A は北大西洋起源深層水の経路、D はより古い深層水である。Ideal Age はその違いを見えるようにする。**  
> **A is the Atlantic deep pathway, and D is older deep water. Ideal Age makes this difference visible.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.8)

## 1. Question: なぜ Ideal Age を計算するのか / Why compute Ideal Age?

保存トレーサーは、水塊がどこへ行くかを見せてくれます。  
Passive tracers show where water masses go.

しかし、保存トレーサーだけでは、

```text
その水がどれくらい古いか
どれくらい長く表層から隔離されていたか
```

は直接分かりません。  
But passive tracers do not directly tell us:

```text
how old the water is
how long it has been isolated from the surface
```

そこで Ideal Age を使います。  
Therefore, we use Ideal Age.

PSLNMAD では、次の問いが重要です。  
In PSLNMAD, the important questions are:

```text
A は D より若いのか？
D はどれくらい表層から隔離されるのか？
ベンチレーションが弱くなると A と D の age はどう変わるのか？
```

```text
Is A younger than D?
How long is D isolated from the surface?
How do ages in A and D change when ventilation weakens?
```

## 2. Ideal Age のルール / Rules of Ideal Age

Ideal Age の計算ルールは非常に単純です。  
The rules for Ideal Age are very simple.

```text
surface boxes P, S, L, N:
    age = 0

interior boxes M, A, D:
    age increases with time
    age is transported by circulation
```

```text
表層 box P, S, L, N:
    age = 0 にリセット

内部 box M, A, D:
    時間とともに age が増える
    age は循環によって輸送される
```

つまり、表層では時計を 0 に戻し、内部では時計を進めます。  
In other words, the clock is reset at the surface and runs in the interior.

## 3. 7-box model の準備 / Prepare 7-box model

06-04 と同じ 7-box 構造を使います。  
We use the same seven-box structure as in 06-04.

```text
P, S, L, N, M, A, D
```

基本経路は、

```text
N → A → D → S → P → L → N
```

です。  
The basic pathway is:

```text
N → A → D → S → P → L → N
```

さらに、

```text
S ↔ M
L ↔ M
```

の交換を入れます。  
We also include exchanges:

```text
S ↔ M
L ↔ M
```

In [ ]:
boxes = ["P", "S", "L", "N", "M", "A", "D"]
surface_boxes = ["P", "S", "L", "N"]
interior_boxes = ["M", "A", "D"]

VOCN_TOTAL = 1.292e18
VOLUME = {
    "P": 0.18 * VOCN_TOTAL,
    "S": 0.05 * VOCN_TOTAL,
    "L": 0.12 * VOCN_TOTAL,
    "N": 0.03 * VOCN_TOTAL,
    "M": 0.27 * VOCN_TOTAL,
    "A": 0.12 * VOCN_TOTAL,
}
VOLUME["D"] = VOCN_TOTAL - sum(VOLUME.values())

volumes = np.array([VOLUME[b] for b in boxes])
idx = {b: i for i, b in enumerate(boxes)}

Q_DEFAULT = 2.0e7
DT = 8.64e4
SEC_PER_YEAR = 365 * 86400
DT_YEAR = DT / SEC_PER_YEAR

pathway = [("N", "A"), ("A", "D"), ("D", "S"), ("S", "P"), ("P", "L"), ("L", "N")]
exchanges_default = [("S", "M", 0.3e7), ("L", "M", 0.2e7)]

pd.DataFrame({
    "Box": boxes,
    "Layer": ["surface", "surface", "surface", "surface", "mid-depth", "Atlantic deep pathway", "deep"],
    "Volume fraction": [VOLUME[b] / VOCN_TOTAL for b in boxes],
})

## 4. 輸送行列を作る / Build the transport matrix

これまでと同じ方法で輸送行列を作ります。  
We build the transport matrix using the same method as before.

In [ ]:
def build_flux_matrix(pathway, Q, boxes):
    F = np.zeros((len(boxes), len(boxes)))
    local_idx = {b: i for i, b in enumerate(boxes)}
    for source, target in pathway:
        i = local_idx[target]
        j = local_idx[source]
        F[i, j] += Q
        F[j, j] -= Q
    return F

def build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges=None):
    F = build_flux_matrix(pathway, Q, boxes)
    local_idx = {b: i for i, b in enumerate(boxes)}
    if exchanges is None:
        exchanges = []
    for a, b, q in exchanges:
        ia = local_idx[a]
        ib = local_idx[b]
        F[ib, ia] += q
        F[ia, ia] -= q
        F[ia, ib] += q
        F[ib, ib] -= q
    return F

F = build_flux_matrix_with_exchange(pathway, Q_DEFAULT, boxes, exchanges_default)
pd.DataFrame(F, index=boxes, columns=boxes)

## 5. 1 step の Ideal Age 更新 / One-step Ideal Age update

Ideal Age の 1 step 更新は、次の順番で行います。  
The one-step update of Ideal Age is:

```text
1. age を輸送する
2. 内部 box の age を dt だけ増やす
3. 表層 box の age を 0 に戻す
```

```text
1. transport age
2. increase age in interior boxes by dt
3. reset age in surface boxes to 0
```

ここで注意する点は、age もトレーサーとして輸送されることです。  
The key point is that age is also transported as a tracer.

In [ ]:
def one_step_transport(c, F):
    return c + (F @ c) * DT / volumes

def one_step_age(age, F):
    new = one_step_transport(age, F)

    # interior boxes become older
    for b in interior_boxes:
        new[idx[b]] += DT_YEAR

    # surface boxes are reset to zero
    for b in surface_boxes:
        new[idx[b]] = 0.0

    return new

age0 = np.zeros(len(boxes))
age1 = one_step_age(age0, F)

pd.DataFrame({
    "Box": boxes,
    "Initial age": age0,
    "After one day": age1,
})

### Mini exercise 1 / ミニ演習 1

表層 box の age が 0 に戻る理由を説明してください。  
Explain why the age in surface boxes is reset to 0.

内部 box の age が増える理由を説明してください。  
Explain why age in interior boxes increases.

## 6. 標準実験 / Standard experiment

まず、標準的な輸送強度で 3000 年積分します。  
First, we integrate for 3000 years with the standard transport strength.

In [ ]:
def run_ideal_age(years=3000, Q=Q_DEFAULT, exchanges=exchanges_default):
    F_local = build_flux_matrix_with_exchange(pathway, Q, boxes, exchanges)
    age = np.zeros(len(boxes))
    hist = []

    for day in range(years * 365 + 1):
        if day % 365 == 0:
            row = {"year": day / 365}
            for i, b in enumerate(boxes):
                row[b] = age[i]
            hist.append(row)
        age = one_step_age(age, F_local)

    return pd.DataFrame(hist)

age_std = run_ideal_age(years=3000)

plt.figure()
for b in boxes:
    plt.plot(age_std["year"], age_std[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Ideal age [yr]")
plt.title("Ideal age in PSLNMAD")
plt.legend()
plt.grid(True)
plt.show()

## 7. A と D を比較する / Compare A and D

PSLNMAD の目的の一つは、A と D を分けて見ることです。  
One purpose of PSLNMAD is to examine A and D separately.

A は N から入る比較的新しい深層経路です。  
A is the relatively young deep pathway receiving water from N.

D は A の下流にある、より古い深層水です。  
D is older deep water downstream of A.

したがって、D の age は A より大きくなると予想されます。  
Therefore, D is expected to be older than A.

In [ ]:
plt.figure()
plt.plot(age_std["year"], age_std["A"], label="A")
plt.plot(age_std["year"], age_std["D"], label="D")
plt.xlabel("Time [yr]")
plt.ylabel("Ideal age [yr]")
plt.title("Ideal age: A vs D")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame({
    "Box": ["M", "A", "D"],
    "Final ideal age [yr]": [
        age_std["M"].iloc[-1],
        age_std["A"].iloc[-1],
        age_std["D"].iloc[-1],
    ],
})

### Mini exercise 2 / ミニ演習 2

A と D のどちらが古くなりましたか。  
Which becomes older, A or D?

その理由を、N → A → D という経路を使って説明してください。  
Explain why using the pathway N → A → D.

## 8. M, A, D の違い / Difference among M, A, and D

PSLNMAD では、内部 box が 3 つあります。  
In PSLNMAD, there are three interior boxes.

```text
M: mid-depth
A: Atlantic deep pathway
D: older deep ocean
```

これらはすべて内部 box ですが、表層とのつながり方が違います。  
They are all interior boxes, but their connections to the surface differ.

M は S や L と交換します。  
M exchanges with S and L.

A は N から沈み込んだ水を受け取ります。  
A receives water sinking from N.

D は A の下流にあります。  
D is downstream of A.

In [ ]:
plt.figure()
for b in ["M", "A", "D"]:
    plt.plot(age_std["year"], age_std[b], label=b)
plt.xlabel("Time [yr]")
plt.ylabel("Ideal age [yr]")
plt.title("Interior ideal age: M, A, D")
plt.legend()
plt.grid(True)
plt.show()

## 9. ベンチレーション強度 Q の感度 / Sensitivity to ventilation strength Q

次に、循環強度 Q を変えます。  
Next, we change circulation strength Q.

Q が大きいほど、水は速く動きます。  
Larger Q means faster water movement.

したがって、内部水は若くなりやすいと予想されます。  
Therefore, interior waters are expected to become younger.

In [ ]:
Q_list = [0.5e7, 1.0e7, 2.0e7, 4.0e7]
summary_Q = []

plt.figure()
for q in Q_list:
    h = run_ideal_age(years=3000, Q=q, exchanges=exchanges_default)
    plt.plot(h["year"], h["D"], label=f"Q={q:.1e}")
    summary_Q.append({
        "Q": q,
        "Final age M": h["M"].iloc[-1],
        "Final age A": h["A"].iloc[-1],
        "Final age D": h["D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Ideal age in D [yr]")
plt.title("Transport strength sensitivity")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_Q)

### Mini exercise 3 / ミニ演習 3

Q を大きくすると、D の age はどう変化しましたか。  
How does age in D change when Q is increased?

これは「ベンチレーションが強い」という言葉とどう対応しますか。  
How does this relate to the phrase "strong ventilation"?

## 10. S-M 交換の感度 / Sensitivity to S-M exchange

M は S と交換しています。  
M exchanges with S.

S-M 交換を強くすると、M はより表層とつながりやすくなります。  
If S-M exchange becomes stronger, M becomes more connected to the surface.

ここでは S-M exchange を変えます。  
Here we vary S-M exchange.

In [ ]:
sm_list = [0.0, 0.1e7, 0.3e7, 0.8e7]
summary_sm = []

plt.figure()
for sm in sm_list:
    exchanges = [("S", "M", sm), ("L", "M", 0.2e7)]
    h = run_ideal_age(years=3000, Q=Q_DEFAULT, exchanges=exchanges)
    plt.plot(h["year"], h["M"], label=f"S-M={sm:.1e}")
    summary_sm.append({
        "S-M exchange": sm,
        "Final age M": h["M"].iloc[-1],
        "Final age A": h["A"].iloc[-1],
        "Final age D": h["D"].iloc[-1],
    })

plt.xlabel("Time [yr]")
plt.ylabel("Ideal age in M [yr]")
plt.title("S-M exchange sensitivity")
plt.legend()
plt.grid(True)
plt.show()

pd.DataFrame(summary_sm)

## 11. A-D 経路の感度 / Sensitivity of A-D pathway

A は N から水を受け取り、D へ送ります。  
A receives water from N and sends it to D.

この経路が強いほど、A と D はより速くベンチレーションされます。  
A stronger pathway ventilates A and D faster.

ここでは、主循環 Q を変えることで A-D 経路の強さを変えています。  
Here, changing Q changes the strength of the A-D pathway.

In [ ]:
plt.figure()
for q in Q_list:
    h = run_ideal_age(years=3000, Q=q, exchanges=exchanges_default)
    plt.plot(h["year"], h["A"], linestyle="--", label=f"A Q={q:.1e}")
    plt.plot(h["year"], h["D"], linestyle="-", label=f"D Q={q:.1e}")
plt.xlabel("Time [yr]")
plt.ylabel("Ideal age [yr]")
plt.title("A-D pathway sensitivity")
plt.legend(ncol=2, fontsize=8)
plt.grid(True)
plt.show()

## 12. Ideal Age と O2・Δ14C への接続 / Connection to O2 and Δ14C

Ideal Age は、それ自体が観測量ではありません。  
Ideal Age itself is not a direct observational quantity.

しかし、O2 や \( \Delta^{14}\mathrm{C} \) の解釈につながります。  
However, it helps interpret O2 and \( \Delta^{14}\mathrm{C} \).

一般に、

```text
古い水:
    O2 が低い
    Δ14C が低い
```

となりやすいです。  
In general:

```text
older water:
    lower O2
    lower Δ14C
```

です。

これは、古い水ほど長く有機物分解を受け、14C も放射壊変するためです。  
This is because older water has experienced organic matter decomposition for longer and radiocarbon has decayed for longer.

In [ ]:
last = age_std.iloc[-1]

# simple diagnostic proxies for teaching
age_values = np.array([last[b] for b in boxes])
O2_proxy = 260.0 - 0.04 * age_values
D14C_proxy = -1000.0 * (1.0 - np.exp(-age_values / 8267.0))

diagnostic = pd.DataFrame({
    "Box": boxes,
    "Ideal age [yr]": age_values,
    "O2 proxy": O2_proxy,
    "Delta14C proxy [permil]": D14C_proxy,
})
diagnostic

In [ ]:
plt.figure()
plt.scatter(diagnostic["Ideal age [yr]"], diagnostic["O2 proxy"])
for _, row in diagnostic.iterrows():
    plt.text(row["Ideal age [yr]"], row["O2 proxy"], row["Box"])
plt.xlabel("Ideal age [yr]")
plt.ylabel("O2 proxy")
plt.title("Older water tends to have lower O2")
plt.grid(True)
plt.show()

plt.figure()
plt.scatter(diagnostic["Ideal age [yr]"], diagnostic["Delta14C proxy [permil]"])
for _, row in diagnostic.iterrows():
    plt.text(row["Ideal age [yr]"], row["Delta14C proxy [permil]"], row["Box"])
plt.xlabel("Ideal age [yr]")
plt.ylabel("Delta14C proxy [permil]")
plt.title("Older water tends to have lower Delta14C")
plt.grid(True)
plt.show()

### Mini exercise 4 / ミニ演習 4

Ideal Age が大きい box では、O2 proxy と Δ14C proxy はどうなりましたか。  
In boxes with larger Ideal Age, what happens to O2 proxy and Δ14C proxy?

この関係は、実際の海洋観測を考えるときにどのように役立つでしょうか。  
How is this relationship useful when thinking about real ocean observations?

## 13. 06-05 のまとめ / Summary of 06-05

この Notebook で学んだことは次です。  
What we learned:

1. Ideal Age は、最後に表層と接してからの時間を表す。  
   Ideal Age represents time since last contact with the surface.

2. 表層 box P, S, L, N では age は 0 にリセットされる。  
   Age is reset to 0 in surface boxes P, S, L, N.

3. 内部 box M, A, D では age が増える。  
   Age increases in interior boxes M, A, D.

4. A は N から入る比較的新しい深層経路であり、D はより古い深層水になりやすい。  
   A is a relatively young deep pathway from N, while D tends to be older deep water.

5. Ideal Age は、O2 や \( \Delta^{14}\mathrm{C} \) の解釈につながる。  
   Ideal Age helps interpret O2 and \( \Delta^{14}\mathrm{C} \).

## Key message

> **PSLNMAD では、A と D の age 差が、北大西洋起源深層水と古い深層水の違いを表す。**  
> **In PSLNMAD, the age difference between A and D expresses the difference between North Atlantic-origin deep water and older deep water.**

## 14. 課題 / Exercises

### 課題 1 / Exercise 1

Ideal Age とは何か説明せよ。  
Explain what Ideal Age is.

### 課題 2 / Exercise 2

表層 box で age を 0 にリセットする理由を説明せよ。  
Explain why age is reset to 0 in surface boxes.

### 課題 3 / Exercise 3

A と D の age が異なる理由を説明せよ。  
Explain why age differs between A and D.

### 課題 4 / Exercise 4

Q を大きくすると、内部 box の age はどう変化するか。  
How does age in interior boxes change when Q is increased?

### 課題 5 / Exercise 5

Ideal Age と O2, \( \Delta^{14}\mathrm{C} \) の関係を説明せよ。  
Explain the relationship between Ideal Age, O2, and \( \Delta^{14}\mathrm{C} \).

## 15. 次へ / Next step

次の Notebook では、7-box model に炭素循環を入れます。  
In the next notebook, we add carbon cycling to the 7-box model.

```text
06-06 Carbon Cycle in 7-box model
```

A と D の DIC, PO4, O2 の違いを調べ、氷期・退氷期 CO2 実験への準備をします。  
We examine differences in DIC, PO4, and O2 between A and D, preparing for glacial-deglacial CO2 experiments.